# Step 7 — Walk-forward ablations: no-graph & motif-augmented

Step 6 showed walk-forward LightGBM on the C1 feature set (108 intrinsic + 103 traj + 17 Block B + 5 PPR) hits F1=0.8557 (vs 0.8146 static).

Two ablations to confirm this is the right operating point:

1. **NG — pure no-graph walk-forward**: 108 intrinsic features only. Confirms graph features still pay off under walk-forward retraining (in the static regime they added +0.013 F1; the question is whether they retain that lift when the model can also learn from labels in the test period).
2. **C1+D — motif-augmented walk-forward**: re-add the 6 Block D 2-hop temporal motif features that hurt under static training. The hypothesis: motifs were noisy under static train because most extreme motif counts came from a few hub wallets in unfamiliar timesteps, but walk-forward retraining sees those patterns evolve and may now extract signal.

## Models
Same LightGBM config as step 6: `learning_rate=0.1, num_leaves=63, min_child_samples=20`, val window `[t* − 5, t* − 1]` for early stopping, refit on full train at val-selected n_estimators. F1 reported at threshold 0.5 (which beat val-tuned threshold in step 6).

## Reference
- Static RF C1: F1=0.8146, AUC=0.9209, PR-AUC=0.8067
- Walk-fwd LGB C1: F1=0.8557, AUC=0.9749, PR-AUC=0.8988

In [1]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, classification_report
import lightgbm as lgb

ROOT = Path.cwd()
while not (ROOT / "transactions_data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
TRANSACTIONS_DIR = ROOT / "transactions_data"
WALLETS_DIR      = ROOT / "actors_data"
CACHE_DIR        = ROOT / "architectures" / "inductive_tx_classification" / "cross_step_tx_graph" / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_END    = 34
N_TIME_STEPS = 49
RANDOM_SEED  = 175
VAL_WIN      = 5
MAX_WALLET_DEGREE = 50
MOTIF_DECAY  = 0.2
np.random.seed(RANDOM_SEED)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/dask/dataframe/__init__.py:49: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [2]:
tx_features_df = pd.read_csv(TRANSACTIONS_DIR / "txs_features.txt")
tx_classes_df  = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.txt")
tx_classes_df["label"] = tx_classes_df["class"].map({1:1,2:0,3:-1}).astype(np.int8)
all_cols = list(tx_features_df.columns)
GRAPH_COLS = [c for c in all_cols if c.startswith("Aggregate_feature")] + ["in_txs_degree","out_txs_degree"]
feat_cols_intr = [c for c in all_cols if c not in ("txId","Time step") and c not in GRAPH_COLS]
F_INTRINSIC = len(feat_cols_intr)
merged = tx_features_df[["txId","Time step"]].merge(tx_classes_df[["txId","label"]], on="txId", how="left")
tx_time = merged["Time step"].astype(np.int64).values
tx_label = merged["label"].fillna(-1).astype(np.int64).values
tx_X_intr = tx_features_df[feat_cols_intr].fillna(0).astype(np.float32).values
tx_id_to_idx = {tid:i for i, tid in enumerate(tx_features_df["txId"].values)}
N_tx = len(tx_features_df)

# Cached features from step 5
traj_X    = np.load(CACHE_DIR / "traj_X.npy")
block_B_X = np.load(CACHE_DIR / "block_B_X.npy")
block_C_X = np.load(CACHE_DIR / "block_C_X.npy")
print(f"  X shapes: intrinsic={tx_X_intr.shape}  traj={traj_X.shape}  B={block_B_X.shape}  C={block_C_X.shape}")

  X shapes: intrinsic=(203769, 108)  traj=(203769, 103)  B=(203769, 17)  C=(203769, 5)


In [3]:
# Block D: 2-hop temporal motifs. Cached on disk after first compute.
D_NPY = CACHE_DIR / "block_D_X.npy"
if D_NPY.exists():
    print("Loading cached Block D...")
    block_D_X = np.load(D_NPY)
else:
    print("Computing Block D motifs...")
    addr_tx_df = pd.read_csv(WALLETS_DIR / "AddrTx_edgelist.txt")
    tx_addr_df = pd.read_csv(WALLETS_DIR / "TxAddr_edgelist.txt")
    wallet_addrs = sorted(set(addr_tx_df["input_address"].unique()) | set(tx_addr_df["output_address"].unique()))
    wallet_addr_to_idx = {a:i for i,a in enumerate(wallet_addrs)}
    addr_tx_df["w"]  = addr_tx_df["input_address"].map(wallet_addr_to_idx)
    addr_tx_df["tx"] = addr_tx_df["txId"].map(tx_id_to_idx)
    addr_tx_df = addr_tx_df.dropna(subset=["w","tx"]).astype({"w":"int64","tx":"int64"})
    tx_addr_df["w"]  = tx_addr_df["output_address"].map(wallet_addr_to_idx)
    tx_addr_df["tx"] = tx_addr_df["txId"].map(tx_id_to_idx)
    tx_addr_df = tx_addr_df.dropna(subset=["w","tx"]).astype({"w":"int64","tx":"int64"})
    wallet_in_txs  = defaultdict(list); wallet_out_txs = defaultdict(list)
    for tx, w in zip(addr_tx_df["tx"].values, addr_tx_df["w"].values):
        wallet_in_txs[int(w)].append((int(tx), int(tx_time[tx])))
    for tx, w in zip(tx_addr_df["tx"].values, tx_addr_df["w"].values):
        wallet_out_txs[int(w)].append((int(tx), int(tx_time[tx])))
    for w in wallet_in_txs:  wallet_in_txs[w].sort(key=lambda r:r[1])
    for w in wallet_out_txs: wallet_out_txs[w].sort(key=lambda r:r[1])
    all_incidence = defaultdict(set)
    for w, lst in wallet_in_txs.items():
        for tx,_ in lst: all_incidence[w].add(tx)
    for w, lst in wallet_out_txs.items():
        for tx,_ in lst: all_incidence[w].add(tx)
    ROLE_OUT_IN, ROLE_OUT_OUT, ROLE_IN_IN, ROLE_IN_OUT = 0,1,2,3
    src_l, dst_l, dt_l, role_l = [], [], [], []
    for w in all_incidence:
        if len(all_incidence[w]) < 2 or len(all_incidence[w]) > MAX_WALLET_DEGREE: continue
        events = []
        for tx, t in wallet_in_txs.get(w, []):  events.append((t, tx, 'in'))
        for tx, t in wallet_out_txs.get(w, []): events.append((t, tx, 'out'))
        events.sort(key=lambda r:r[0])
        K = len(events)
        for i in range(K):
            ti, txi, si = events[i]
            for j in range(i+1, K):
                tj, txj, sj = events[j]
                if tj <= ti or txi == txj: continue
                if   si=='out' and sj=='in':  role = ROLE_OUT_IN
                elif si=='out' and sj=='out': role = ROLE_OUT_OUT
                elif si=='in'  and sj=='in':  role = ROLE_IN_IN
                else:                          role = ROLE_IN_OUT
                src_l.append(txi); dst_l.append(txj); dt_l.append(tj-ti); role_l.append(role)
    src = np.array(src_l, dtype=np.int64); dst = np.array(dst_l, dtype=np.int64)
    dt_edges = np.array(dt_l, dtype=np.int32); role_e = np.array(role_l, dtype=np.int8)
    order_in = np.argsort(dst, kind="stable")
    src_in = src[order_in]; role_in = role_e[order_in]
    in_offsets = np.searchsorted(dst[order_in], np.arange(N_tx + 1)).astype(np.int64)
    block_D_X = np.zeros((N_tx, 6), dtype=np.float32)
    t0 = time.time()
    for j in range(N_tx):
        a, b = in_offsets[j], in_offsets[j+1]
        if a == b: continue
        t_T = int(tx_time[j])
        primes = src_in[a:b]; role_first = role_in[a:b]
        total=0; cnt_t2pp_ill=0; cnt_t1p_ill=0; cnt_both_outin=0; cnt_recent=0; dec_t2pp_ill=0.0
        for k in range(b-a):
            T1 = int(primes[k]); r1 = int(role_first[k])
            a2, b2 = in_offsets[T1], in_offsets[T1+1]
            if a2 == b2: continue
            T2pp = src_in[a2:b2]; r2 = role_in[a2:b2]
            t_T2pp = tx_time[T2pp]
            valid = (T2pp != j) & (t_T2pp < int(tx_time[T1]))
            if not valid.any(): continue
            T2pp_v = T2pp[valid]; r2_v = r2[valid]; tT2_v = t_T2pp[valid]
            n_v = T2pp_v.size; total += n_v
            ill_T2pp_mask = (tx_label[T2pp_v]==1)
            cnt_t2pp_ill += int(ill_T2pp_mask.sum())
            cnt_t1p_ill += n_v if tx_label[T1]==1 else 0
            cnt_both_outin += int(((r2_v==ROLE_OUT_IN) & (r1==ROLE_OUT_IN)).sum())
            cnt_recent += int(((t_T - tT2_v) <= 5).sum())
            if ill_T2pp_mask.any():
                dt2 = (t_T - tT2_v[ill_T2pp_mask]).astype(np.float64)
                dec_t2pp_ill += float(np.exp(-MOTIF_DECAY*dt2).sum())
        block_D_X[j,0]=total; block_D_X[j,1]=cnt_t2pp_ill; block_D_X[j,2]=cnt_t1p_ill
        block_D_X[j,3]=cnt_both_outin; block_D_X[j,4]=cnt_recent; block_D_X[j,5]=dec_t2pp_ill
    np.save(D_NPY, block_D_X)
    print(f"  Block D done in {time.time()-t0:.0f}s. cached.")
print(f"  block_D_X={block_D_X.shape}")
print("  motif sparsity (full set):")
for i, nm in enumerate(["motif2_total","motif2_T2pp_illicit","motif2_T1p_illicit","motif2_both_outin","motif2_recent_le5","motif2_decayed_T2pp_illicit"]):
    nz = int((block_D_X[:,i]>0).sum())
    print(f"    {nm:30s}  nonzero={nz:>6,}  max={block_D_X[:,i].max():.2f}")

Computing Block D motifs...


  Block D done in 5s. cached.
  block_D_X=(203769, 6)
  motif sparsity (full set):
    motif2_total                    nonzero=22,819  max=68497296.00
    motif2_T2pp_illicit             nonzero= 1,150  max=2868.00
    motif2_T1p_illicit              nonzero=   196  max=90.00
    motif2_both_outin               nonzero= 4,460  max=1077.00
    motif2_recent_le5               nonzero=13,874  max=763714.00
    motif2_decayed_T2pp_illicit     nonzero= 1,150  max=42.98


In [4]:
X_NG     = tx_X_intr                                                              # 108
X_C1     = np.concatenate([tx_X_intr, traj_X, block_B_X, block_C_X], axis=1)      # 233
X_C1D    = np.concatenate([X_C1, block_D_X], axis=1)                              # 239
print(f"  X_NG={X_NG.shape}  X_C1={X_C1.shape}  X_C1D={X_C1D.shape}")
labelled = (tx_label != -1)
test_mask = labelled & (tx_time > TRAIN_END)
y_test = tx_label[test_mask]; test_t_arr = tx_time[test_mask]
print(f"  test n={int(test_mask.sum()):,}  illicit={int(y_test.sum()):,}")

  X_NG=(203769, 108)  X_C1=(203769, 233)  X_C1D=(203769, 239)
  test n=16,670  illicit=1,083


In [5]:
def walk_forward_lgb(X, name):
    print(f"\n=== Walk-forward LightGBM [{name}] (dim={X.shape[1]}) ===")
    wf_p  = np.full(len(tx_time), np.nan, dtype=np.float64)
    iters_log = {}
    t0 = time.time()
    for tstar in range(TRAIN_END+1, N_TIME_STEPS+1):
        tr_es_mask = labelled & (tx_time <= tstar - VAL_WIN - 1)
        val_mask   = labelled & (tx_time >= tstar - VAL_WIN) & (tx_time <= tstar - 1)
        full_mask  = labelled & (tx_time <= tstar - 1)
        te_mask    = labelled & (tx_time == tstar)
        if int(te_mask.sum()) == 0: continue
        if int(val_mask.sum()) >= 50 and int(tr_es_mask.sum()) >= 100:
            y_tr_es = tx_label[tr_es_mask]; y_val = tx_label[val_mask]
            pos_w_es = (1.0-y_tr_es.mean())/max(y_tr_es.mean(), 1e-6)
            m_es = lgb.LGBMClassifier(
                objective="binary", n_estimators=2000,
                learning_rate=0.1, num_leaves=63, min_child_samples=20,
                subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0,
                scale_pos_weight=pos_w_es, verbose=-1, n_jobs=-1, random_state=RANDOM_SEED,
            )
            m_es.fit(X[tr_es_mask], y_tr_es,
                     eval_set=[(X[val_mask], y_val)],
                     callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)])
            n_iter = max(int(m_es.best_iteration_ or 200), 50)
        else:
            n_iter = 200
        iters_log[tstar] = n_iter
        y_full = tx_label[full_mask]
        pos_w_full = (1.0-y_full.mean())/max(y_full.mean(), 1e-6)
        m_full = lgb.LGBMClassifier(
            objective="binary", n_estimators=n_iter,
            learning_rate=0.1, num_leaves=63, min_child_samples=20,
            subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0,
            scale_pos_weight=pos_w_full, verbose=-1, n_jobs=-1, random_state=RANDOM_SEED,
        )
        m_full.fit(X[full_mask], y_full)
        p_te = m_full.predict_proba(X[te_mask])[:,1]
        wf_p[te_mask] = p_te
        n_pos = int(tx_label[te_mask].sum())
        f1_05 = f1_score(tx_label[te_mask], (p_te>=0.5).astype(int), pos_label=1, zero_division=0) if n_pos>0 else float('nan')
        print(f"  t*={tstar:>2d}  n_full={int(full_mask.sum()):>5,}  n_te={int(te_mask.sum()):>4,}  ill={n_pos:>3d}  iters={n_iter:>3d}  F1@0.5={f1_05:.4f}")
    print(f"  total: {time.time()-t0:.0f}s")
    return wf_p

# Run all three configs
wf_p_NG  = walk_forward_lgb(X_NG,  "NG  (108 intrinsic, no graph)")
wf_p_C1  = walk_forward_lgb(X_C1,  "C1  (108 + 103 traj + 17 B + 5 PPR = 233)")
wf_p_C1D = walk_forward_lgb(X_C1D, "C1D (C1 + 6 motifs = 239)")


=== Walk-forward LightGBM [NG  (108 intrinsic, no graph)] (dim=108) ===


  t*=35  n_full=29,894  n_te=1,341  ill=182  iters=132  F1@0.5=0.9402


  t*=36  n_full=31,235  n_te=1,708  ill= 33  iters=136  F1@0.5=0.9429


  t*=37  n_full=32,943  n_te= 498  ill= 40  iters=180  F1@0.5=0.7887


  t*=38  n_full=33,441  n_te= 756  ill=111  iters=109  F1@0.5=0.9143


...

  t*=45  n_full=42,838  n_te=1,221  ill=  5  iters= 50  F1@0.5=0.0000


  t*=46  n_full=44,059  n_te= 712  ill=  2  iters= 95  F1@0.5=0.6667


  t*=47  n_full=44,771  n_te= 846  ill= 22  iters= 91  F1@0.5=0.1429


  t*=48  n_full=45,617  n_te= 471  ill= 36  iters=105  F1@0.5=0.5200


  t*=49  n_full=46,088  n_te= 476  ill= 56  iters= 78  F1@0.5=0.9550
  total: 72s


In [6]:
def eval_summary(name, wf_p):
    p = wf_p[test_mask]
    yp_05 = (p >= 0.5).astype(int)
    f1 = f1_score(y_test, yp_05, zero_division=0)
    auc = roc_auc_score(y_test, p)
    pr = average_precision_score(y_test, p)
    cliff_mask = test_t_arr >= 43
    f1_cliff = f1_score(y_test[cliff_mask], yp_05[cliff_mask], zero_division=0) if y_test[cliff_mask].sum() > 0 else float('nan')
    return f1, auc, pr, f1_cliff, yp_05, p

rows = []
preds = {}
for name, wf_p in [("NG (108)", wf_p_NG), ("C1 (233)", wf_p_C1), ("C1D (239)", wf_p_C1D)]:
    f1, auc, pr, f1_cliff, yp_05, p = eval_summary(name, wf_p)
    rows.append((name, f1, auc, pr, f1_cliff))
    preds[name] = (yp_05, p)

print("="*78)
print(f"{'Walk-forward LGB ablation':35s}  {'F1':>7s}  {'AUC':>7s}  {'PR-AUC':>7s}  {'F1[t≥43]':>9s}")
print("="*78)
for n, f1, auc, pr, f1c in rows:
    print(f"  {n:33s}  {f1:>7.4f}  {auc:>7.4f}  {pr:>7.4f}  {f1c:>9.4f}")

f1_C1 = next(r[1] for r in rows if r[0].startswith("C1 "))
f1_NG = next(r[1] for r in rows if r[0].startswith("NG"))
f1_C1D = next(r[1] for r in rows if r[0].startswith("C1D"))
print()
print(f"  Δ F1 (C1 − NG):    {f1_C1-f1_NG:+.4f}    [does the graph help under walk-forward?]")
print(f"  Δ F1 (C1D − C1):   {f1_C1D-f1_C1:+.4f}    [do motifs help once retraining is rolling?]")

# Per-timestep
print("\n"+"="*78)
print("Per-timestep F1@0.5 (illicit class)")
print("="*78)
print(f"{'t':>3}  {'n':>5}  {'ill':>4}  {'NG':>8}  {'C1':>8}  {'C1D':>8}")
for tstar in range(TRAIN_END+1, N_TIME_STEPS+1):
    m = (test_t_arr == tstar)
    if not m.any(): continue
    yt = y_test[m]
    if yt.sum() == 0: f_n=f_c=f_d=float('nan')
    else:
        f_n = f1_score(yt, preds["NG (108)"][0][m], zero_division=0)
        f_c = f1_score(yt, preds["C1 (233)"][0][m], zero_division=0)
        f_d = f1_score(yt, preds["C1D (239)"][0][m], zero_division=0)
    print(f"{tstar:>3}  {int(m.sum()):>5}  {int(yt.sum()):>4}  {f_n:>8.4f}  {f_c:>8.4f}  {f_d:>8.4f}")

Walk-forward LGB ablation                 F1      AUC   PR-AUC   F1[t≥43]
  NG (108)                            0.8055   0.9700   0.8712     0.3925
  C1 (233)                            0.8557   0.9749   0.8988     0.5357
  C1D (239)                           0.8525   0.9760   0.9023     0.5505

  Δ F1 (C1 − NG):    +0.0502    [does the graph help under walk-forward?]
  Δ F1 (C1D − C1):   -0.0032    [do motifs help once retraining is rolling?]

Per-timestep F1@0.5 (illicit class)
  t      n   ill        NG        C1       C1D
 35   1341   182    0.9402    0.9725    0.9725
 36   1708    33    0.9429    0.9706    0.9851
 37    498    40    0.7887    0.7879    0.7879
 38    756   111    0.9143    0.9378    0.9275
 39   1183    81    0.9200    0.9419    0.9359
 40   1211   112    0.7437    0.7677    0.7653
 41   1132   116    0.9231    0.9356    0.9270
 42   2154   239    0.8468    0.8821    0.8729
 43   1370    24    0.1333    0.1622    0.1818
 44   1591    24    0.2609    0.3721    0.325

In [7]:
print("="*78)
print("FULL HISTORY OF THE PIPELINE (test t in [35, 49])")
print("="*78)
history = [
    ("STATIC RF [108 intrinsic, no graph]",          0.8021, 0.9026, 0.7855),
    ("STATIC RF [108 + 103 traj]",                    0.8098, 0.9196, 0.8029),
    ("STATIC RF [+ Block B pair feats]",              0.8122, 0.9160, 0.8027),
    ("STATIC RF [+ PPR illicit-seeded] (C1)",         0.8149, 0.9209, 0.8067),
    ("STATIC LGB C1",                                  0.8004, 0.9279, 0.8041),
    ("WALK-FWD RF C1",                                 0.8240, 0.9721, 0.8762),
    ("WALK-FWD LGB C1 @0.5 (step 6)",                 0.8557, 0.9749, 0.8988),
    ("WALK-FWD LGB NG (no graph)",                    rows[0][1], rows[0][2], rows[0][3]),
    ("WALK-FWD LGB C1 (re-run)",                      rows[1][1], rows[1][2], rows[1][3]),
    ("WALK-FWD LGB C1+D (with motifs)",               rows[2][1], rows[2][2], rows[2][3]),
]
print(f"{'Setup':45s}  {'F1':>7s}  {'AUC':>7s}  {'PR-AUC':>7s}")
for name, f1, auc, pr in history:
    print(f"  {name:43s}  {f1:>7.4f}  {auc:>7.4f}  {pr:>7.4f}")

FULL HISTORY OF THE PIPELINE (test t in [35, 49])
Setup                                               F1      AUC   PR-AUC
  STATIC RF [108 intrinsic, no graph]           0.8021   0.9026   0.7855
  STATIC RF [108 + 103 traj]                    0.8098   0.9196   0.8029
  STATIC RF [+ Block B pair feats]              0.8122   0.9160   0.8027
  STATIC RF [+ PPR illicit-seeded] (C1)         0.8149   0.9209   0.8067
  STATIC LGB C1                                 0.8004   0.9279   0.8041
  WALK-FWD RF C1                                0.8240   0.9721   0.8762
  WALK-FWD LGB C1 @0.5 (step 6)                 0.8557   0.9749   0.8988
  WALK-FWD LGB NG (no graph)                    0.8055   0.9700   0.8712
  WALK-FWD LGB C1 (re-run)                      0.8557   0.9749   0.8988
  WALK-FWD LGB C1+D (with motifs)               0.8525   0.9760   0.9023
